# The Machine Learning Workflow

In the explainer we just read, we saw that machine learning is pattern recognition at scale — and that we have already been doing it. The regression model we built in Unit 8 learned from historical data and made predictions on new observations. That is the core of ML.

Now we put that idea into practice. The compliance team at Northgate Retail Bank wants an automated fraud alert system, and we are going to build the first version. This notebook walks through the complete ML workflow on that problem: from defining what we are predicting, through exploring the data, to training a model and discovering why the most obvious evaluation metric can be dangerously misleading.

By the end of this notebook we will be able to:

- Define a machine learning problem in terms of **features** and a **target**
- Explore data to understand class distributions and feature patterns
- Split data into **training** and **test** sets
- Fit a **logistic regression** model and generate predictions
- Evaluate model performance and explain why **accuracy** can be misleading

In [ ]:
%pip install -q pandas matplotlib seaborn scikit-learn

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

sns.set_style("whitegrid")

---

## 1. Define the problem

Every ML project starts with a clear question. Ours comes straight from the bank's compliance team:

> **Given a transaction's attributes, can we predict whether it is fraudulent?**

This is a **binary classification** problem. The **target variable** is `is_fraud` (True or False) — the column that contains the answer we want to predict. Everything else we use to make that prediction is a **feature**. If the terminology feels familiar, it should: in the explainer, we saw that "target" is the ML term for what statistics calls the outcome variable, and "features" are what we have been calling independent variables.

Let's load the data and see what we are working with.

In [ ]:
df = pd.read_csv("../data/transactions.csv")
print(f"{len(df):,} transactions")
df.head()

In [ ]:
df.info()

---

## 2. Explore the data

Before modelling, we need to understand what we are working with. The first thing to check in any classification problem is the **class distribution**: how many examples of each outcome does the dataset contain?

In [ ]:
fraud_counts = df["is_fraud"].value_counts()
print(fraud_counts)
print(f"\nFraud rate: {df['is_fraud'].mean():.1%}")

Only about 3% of transactions are fraud. This is an **imbalanced dataset**, typical of fraud detection in the real world. The vast majority of transactions are legitimate. Keep this number in mind; it will matter a great deal when we evaluate the model later.

Now let's see whether the features look different for fraudulent versus legitimate transactions. If fraud transactions have distinctive patterns, our model has something to learn from.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Amount
for label, group in df.groupby("is_fraud"):
    axes[0].hist(group["amount"], bins=40, alpha=0.5,
                 label="Fraud" if label else "Legitimate")
axes[0].set_xlabel("Amount (£)")
axes[0].set_ylabel("Count")
axes[0].set_title("Transaction amount")
axes[0].legend()

# Hour of day
for label, group in df.groupby("is_fraud"):
    axes[1].hist(group["hour_of_day"], bins=24, alpha=0.5, density=True,
                 label="Fraud" if label else "Legitimate")
axes[1].set_xlabel("Hour of day")
axes[1].set_ylabel("Density")
axes[1].set_title("Transaction hour")
axes[1].legend()

# Distance from home
for label, group in df.groupby("is_fraud"):
    axes[2].hist(group["distance_from_home_km"], bins=40, alpha=0.5, density=True,
                 label="Fraud" if label else "Legitimate")
axes[2].set_xlabel("Distance from home (km)")
axes[2].set_ylabel("Density")
axes[2].set_title("Distance from home")
axes[2].legend()

plt.tight_layout()
plt.show()

Good news: there are visible differences. Fraudulent transactions tend to involve higher amounts, occur more often in the early hours, and happen further from the customer's home. But notice the overlap between the two distributions in every chart. No single feature cleanly separates fraud from legitimate activity, which is exactly why we need a model that can consider multiple features together.

In [ ]:
# International transaction rate by fraud status
print(df.groupby("is_fraud")["is_international"].mean())

International transactions are far more common among fraud cases. Another useful signal for the model.

---

## 3. Select features

We saw in the explainer that ML models learn from features — the input columns that carry predictive information. We need to choose which columns to use and ensure they are in numeric format, because scikit-learn works with numbers.

We will start with four features that showed clear differences between fraud and legitimate transactions: `amount`, `hour_of_day`, `is_international`, and `distance_from_home_km`.

In [ ]:
features = ["amount", "hour_of_day", "is_international", "distance_from_home_km"]

X = df[features].copy()
y = df["is_fraud"].astype(int)

# is_international is boolean; convert to 0/1
X["is_international"] = X["is_international"].astype(int)

print(f"Features shape: {X.shape}")
print(f"Target shape:   {y.shape}")
X.head()

`X` is the **feature matrix** (the inputs) and `y` is the **target vector** (the labels we want to predict). This `X` / `y` naming convention is standard in scikit-learn and we will see it throughout these notebooks.

---

## 4. Split into training and test sets

This step should feel familiar from Unit 8. We must evaluate the model on data it has **never seen during training**. Otherwise we cannot tell whether it has learnt genuine patterns or simply memorised the training examples. That is the overfitting problem we already know well.

The standard approach is a **train/test split**: hold back a portion of the data (here, 25%) for testing.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

print(f"Training set: {len(X_train):,} transactions")
print(f"Test set:     {len(X_test):,} transactions")
print(f"\nFraud rate in training: {y_train.mean():.1%}")
print(f"Fraud rate in test:     {y_test.mean():.1%}")

The `stratify=y` argument ensures both sets have the same fraud rate as the original data. Without stratification, the small number of fraud cases might end up unevenly distributed between the two sets.

---

## 5. Fit a logistic regression model

Logistic regression is a natural starting point. We already know linear regression from Unit 8; logistic regression uses the same idea but outputs a probability of belonging to a class rather than a continuous number. It is a solid baseline model for classification.

Scikit-learn follows a consistent three-step API:
1. Create the model object
2. Call `.fit(X_train, y_train)` to train it
3. Call `.predict(X_test)` to generate predictions

This pattern is the same for every model in the library. Learn it once and we can swap in a different algorithm with a single line change.

In [ ]:
model = LogisticRegression(random_state=42, max_iter=1000)
model.fit(X_train, y_train)

print("Model trained.")

In [ ]:
y_pred = model.predict(X_test)

print(f"Predictions: {y_pred[:20]}")
print(f"Actual:      {y_test.values[:20]}")

---

## 6. Evaluate with accuracy

The simplest evaluation metric is **accuracy**: the proportion of predictions that are correct. Let's see how our model does.

In [ ]:
acc = accuracy_score(y_test, y_pred)
print(f"Accuracy: {acc:.2%}")

### Why accuracy is misleading here

That number looks impressive. But remember the 3% fraud rate? Let's see what happens if we build the laziest possible model — one that **always predicts "not fraud"**.

In [ ]:
import numpy as np

# A model that always predicts 0 (not fraud)
y_pred_naive = np.zeros(len(y_test), dtype=int)
naive_acc = accuracy_score(y_test, y_pred_naive)
print(f"Naive model accuracy: {naive_acc:.2%}")

A model that does nothing useful still achieves ~97% accuracy, because only ~3% of transactions are fraud. Accuracy rewards the model for getting the easy majority class right while hiding its failures on the rare class that actually matters.

The explainer warned that accuracy alone is often misleading for imbalanced problems. Now we can see why. The real questions for the compliance team are:

- **Of the fraud cases, how many did we catch?** (recall)
- **Of the transactions we flagged as fraud, how many actually were?** (precision)

We will tackle these properly in the next notebook, where we build a more effective classifier.

---

## 7. Exercises

### Exercise 1: Explore merchant categories

Calculate the fraud rate for each `merchant_category` in the original dataframe. Which category has the highest fraud rate? Use `groupby` and the mean of `is_fraud`.

In [ ]:
# Your code here


### Exercise 2: Different test size

Re-run the train/test split with `test_size=0.40` instead of 0.25. Train a new logistic regression on this smaller training set and compute accuracy on the larger test set. How does the result compare?

In [ ]:
# Your code here


### Exercise 3: Add more features

Extend the feature list to include `merchant_category` and `day_of_week`. You will need to convert these categorical columns into numeric form using `pd.get_dummies()`. Train a logistic regression with the expanded features and compare accuracy to the original model.

In [ ]:
# Your code here


### Exercise 4: Count the errors

Using the original model's predictions (`y_pred`), count how many fraud transactions were **missed** (predicted not fraud but actually fraud) and how many legitimate transactions were **incorrectly flagged** (predicted fraud but actually legitimate). What do these numbers tell you about the model's usefulness?

In [ ]:
# Your code here


---

## Summary

In this notebook we worked through the full ML workflow on a real problem: predicting fraudulent transactions for Northgate Retail Bank. We:

- Defined a **binary classification** problem with a clear target variable
- Explored the data and found an **imbalanced** class distribution (~3% fraud)
- Selected numeric **features** and prepared them for modelling
- Split data into **training** and **test** sets with stratification
- Trained a **logistic regression** model using scikit-learn's fit/predict API
- Evaluated with **accuracy** and discovered why it is misleading for rare events

The workflow itself — define, explore, prepare, split, train, evaluate — is the same regardless of the algorithm. What changes is how we measure success. In the next notebook, we will use the confusion matrix and better metrics to build a classifier the compliance team can actually trust.